# MDDB workflow task dependencies

Explore dependencies from left to right. Colors identify input files, configuration flags, derived data, preparation steps, mappings, analyses, and generated files that are consumed downstream. Arrows use the color of the node they leave. Hover over any node to inspect its inputs and outputs.

In [ ]:
import ipywidgets as widgets
from IPython.display import HTML, clear_output, display

from mddb_workflow.resources.task_dependencies import (
    build_dependency_graph,
    create_dependency_figure,
    graph_statistics,
    task_options,
)

In [ ]:
scope_picker = widgets.Dropdown(
    options=[
        ('Entire workflow', 'all'),
        ('Project tasks', 'project'),
        ('MD tasks', 'md'),
    ],
    value='all',
    description='Workflow level',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='23%'),
)
focus_picker = widgets.Dropdown(
    options=[('All tasks', ''), *task_options('all').items()],
    value='',
    description='Focus on a task or connected output',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='52%'),
)
group_sources_toggle = widgets.Checkbox(
    value=True,
    description='Group related inputs and flags',
    indent=False,
    layout=widgets.Layout(width='25%'),
)
graph_output = widgets.Output()
_updating_controls = False


def _render_graph(*_):
    global workflow_graph, dependency_figure
    if _updating_controls:
        return
    workflow_graph = build_dependency_graph(
        scope=scope_picker.value,
        group_sources=group_sources_toggle.value,
        focus=focus_picker.value or None,
    )
    dependency_figure = create_dependency_figure(workflow_graph)
    statistics = graph_statistics(workflow_graph)
    group_counts = ' &middot; '.join(
        f'{count} {group.lower()}'
        for group, count in statistics['task_groups'].items()
    )
    summary = (
        f"<b>{statistics['tasks']} tasks</b> &middot; "
        f"<b>{statistics['outputs']} outputs</b> &middot; "
        f"<b>{statistics['dependencies']} dependency sources</b> &middot; "
        f"<b>{statistics['edges']} relationships</b><br>{group_counts}"
    )
    with graph_output:
        clear_output(wait=True)
        display(HTML(summary))
        dependency_figure.show(
            config={
                'displaylogo': False,
                'responsive': True,
                'scrollZoom': True,
            }
        )


def _change_scope(change):
    global _updating_controls
    _updating_controls = True
    try:
        focus_picker.options = [
            ('All tasks', ''),
            *task_options(change['new']).items(),
        ]
        focus_picker.value = ''
    finally:
        _updating_controls = False
    _render_graph()


scope_picker.observe(_change_scope, names='value')
focus_picker.observe(_render_graph, names='value')
group_sources_toggle.observe(_render_graph, names='value')

controls = widgets.HBox(
    [scope_picker, focus_picker, group_sources_toggle],
    layout=widgets.Layout(width='100%', align_items='center'),
)
display(controls, graph_output)
_render_graph()

Circles are workflow tasks. Squares are grouped inputs or flags; diamonds are individual or derived data dependencies; hexagons are output files used by downstream tasks or values. Legend entries can be toggled, and the graph supports pan, zoom, and hover inspection. In focused views, only the selected node and its direct connections remain at full opacity.